# PlanMargin — reproducible TensorRT qualification

This notebook downloads PlanMargin's versioned model-only release, builds FP32 and FP16 TensorRT engines, measures CUDA-event latency and numerical parity, and compiles the C++17 runner. Use a free **T4 GPU** runtime. No WOMD records are downloaded or redistributed; model quality comes from the separately sealed real-WOMD holdout report.

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], check=True, capture_output=True, text=True).stdout.strip()
print(gpu)

In [ ]:
!git clone --depth 1 https://github.com/ethanvillalovoz/planmargin.git
%cd planmargin
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = f"{os.path.expanduser('~')}/.local/bin:{os.environ['PATH']}"
!uv python install 3.11
!uv sync --frozen --extra nvidia
!uv pip install --python .venv/bin/python tensorrt-cu12==11.2.1.2

## Fetch and verify the deployable model

The model-only release is public and hash-pinned. It contains weights, ONNX, and aggregate training metrics—never source scenarios or per-record outputs.

In [ ]:
!mkdir -p artifacts/experiment-v4/torch-trajectory-model
!curl -L --fail --output artifacts/experiment-v4/torch-trajectory-model/trajectory-model.pmtorch https://github.com/ethanvillalovoz/planmargin/releases/download/trajectory-model-v1/trajectory-model.pmtorch
!curl -L --fail --output artifacts/experiment-v4/torch-trajectory-model/trajectory-model.onnx https://github.com/ethanvillalovoz/planmargin/releases/download/trajectory-model-v1/trajectory-model.onnx
!curl -L --fail --output artifacts/experiment-v4/torch-trajectory-model/training-report.json https://github.com/ethanvillalovoz/planmargin/releases/download/trajectory-model-v1/training-report.json
!printf '%s  %s\n%s  %s\n%s  %s\n' '7b16c2335660097918c2a924b1c0653e85433e18eeaadcdf8d20ac893573b822' 'artifacts/experiment-v4/torch-trajectory-model/trajectory-model.pmtorch' '8ce5582c2a56014f4cd7b78515e92a5298660cd9e7782db56d7b936288c9d813' 'artifacts/experiment-v4/torch-trajectory-model/trajectory-model.onnx' '20a04f22b4d8083f5f4c425353ec4b9c50a177257e77496a17eea9f5307bb159' 'artifacts/experiment-v4/torch-trajectory-model/training-report.json' | sha256sum --check

In [ ]:
!.venv/bin/planmargin-qualify-tensorrt --warmup 50 --iterations 500 --batches 1 8 256
!jq '{status,gates,gpu,environment,measurement,engines}' artifacts/experiment-v4/tensorrt-qualification/qualification-report.json

## Compile and cross-check the C++17 runtime

TensorRT's pip package supplies runtime libraries but not C++ headers. The next cell pins NVIDIA's matching `v11.2` headers, builds the checked-in runner, and measures the same engine.

In [ ]:
import pathlib
import shutil
root = pathlib.Path('/tmp/planmargin-tensorrt')
shutil.rmtree(root, ignore_errors=True)
(root / 'lib').mkdir(parents=True)
!git clone --branch v11.2 --depth 1 https://github.com/NVIDIA/TensorRT.git /tmp/TensorRT
shutil.copytree('/tmp/TensorRT/include', root / 'include')
venv_site = pathlib.Path('.venv/lib/python3.11/site-packages').resolve()
for library in (venv_site / 'tensorrt_libs').glob('libnvinfer.so*'):
    (root / 'lib' / library.name).symlink_to(library)
print(root)

In [ ]:
!cmake -S cpp/tensorrt -B build/tensorrt -DTENSORRT_ROOT=/tmp/planmargin-tensorrt -DCMAKE_BUILD_TYPE=Release
!cmake --build build/tensorrt --parallel
!LD_LIBRARY_PATH=/tmp/planmargin-tensorrt/lib build/tensorrt/planmargin_tensorrt_runner --engine artifacts/experiment-v4/tensorrt-qualification/trajectory-fp32.engine --batch 1 --warmup 50 --iterations 500 | tee artifacts/experiment-v4/tensorrt-qualification/cpp-fp32-batch1.json

In [ ]:
!uv pip freeze --python .venv/bin/python > artifacts/experiment-v4/tensorrt-qualification/environment.lock.txt
!sha256sum artifacts/experiment-v4/torch-trajectory-model/trajectory-model.onnx artifacts/experiment-v4/tensorrt-qualification/trajectory-*.engine > artifacts/experiment-v4/tensorrt-qualification/artifact-sha256.txt
!zip -j artifacts/experiment-v4/planmargin-tensorrt-aggregate.zip artifacts/experiment-v4/tensorrt-qualification/qualification-report.json artifacts/experiment-v4/tensorrt-qualification/cpp-fp32-batch1.json artifacts/experiment-v4/tensorrt-qualification/environment.lock.txt artifacts/experiment-v4/tensorrt-qualification/artifact-sha256.txt
from google.colab import files
files.download('artifacts/experiment-v4/planmargin-tensorrt-aggregate.zip')